# Upload a Harbor dataset and run an evaluation

Publish three local tasks as a versioned taskset, run them with Harbor's built-in Codex agent against the OpenAI API, and inspect the saved results. The notebook stores the API key in NeMo Platform Secrets and passes only its secret reference to the evaluator job.

| Task | What to expect |
| --- | --- |
| `greet-universe` | Codex writes the exact greeting; reward `1.0` is expected. |
| `sum-three` | Codex computes the sum and writes the expected file; reward `1.0` is expected. |
| `debug-agent-runtime-error` | Harbor records an intentional step-level agent timeout plus reward `0.0`; its second step must not run. |

## Before you start

- Use Python 3.12+ with the evaluator's Harbor extra and Jupyter installed in your repository environment.
- Follow the repository [setup guide](../../../../SETUP.md) to start Files, Entities, Evaluator, and Secrets services plus a host-subprocess evaluator worker. Docker must be available to the worker.
- Export `OPENAI_API_KEY` before starting Jupyter. The task containers must be able to reach `api.openai.com`. No local inference model registration or `OPENAI_BASE_URL` override is needed.
- Open this notebook from its own directory or elsewhere in the repository. Run cells in order.

Uploads create persistent Tasks, a Taskset, and Files objects. The notebook registers the current local task content with replacement revisions when the names already exist; submitting a job always starts a new evaluation.


In [ ]:
import io
import json
import os
import re
import tarfile
import time
from pathlib import Path

from IPython.display import Markdown, display
from nemo_evaluator.sdk.harbor import upload_harbor_dataset
from nemo_platform_plugin.client.client import NemoClient
from nemo_platform_plugin.client.errors import ConflictError
from nemo_platform_plugin.evaluator.client import EvaluatorClient
from nemo_platform_plugin.evaluator.types import SubmitAgentEvalJobRequest
from nemo_platform_plugin.secrets.client import SecretsClient
from nemo_platform_plugin.secrets.types import PlatformSecretCreateRequest, PlatformSecretUpdateRequest
from pydantic import SecretStr

# Edit these values, or set the corresponding environment variables.
BASE_URL = os.environ.get("NMP_BASE_URL", "http://localhost:8080")
WORKSPACE = os.environ.get("NMP_WORKSPACE", "default")
TASKSET_NAME = "hello-harbor-taskset"
MODEL_NAME = os.environ.get("HARBOR_CODEX_MODEL", "gpt-5.6-luna")
CODEX_VERSION = os.environ.get("HARBOR_CODEX_VERSION", "0.153.0")
OPENAI_KEY_SECRET = "harbor-openai-api-key"
openai_api_key = os.environ.get("OPENAI_API_KEY")
if not openai_api_key:
    raise EnvironmentError("Set OPENAI_API_KEY before running this notebook.")

cwd = Path.cwd().resolve()
EXAMPLE_DIR = next(
    (
        candidate
        for parent in (cwd, *cwd.parents)
        for candidate in (parent, parent / "plugins/nemo-evaluator/examples/harbor_taskset")
        if (candidate / "harbor_dataset").is_dir() and (candidate / "agent/harbor_wrapper.py").is_file()
    ),
    None,
)
if EXAMPLE_DIR is None:
    raise FileNotFoundError("Open this notebook from the example directory or repository checkout.")
DATASET = EXAMPLE_DIR / "harbor_dataset"

client = NemoClient(
    base_url=BASE_URL,
    workspace=WORKSPACE,
    auth=os.environ.get("NMP_API_KEY"),  # Only needed when your platform requires authentication.
)
evaluator_client = EvaluatorClient.from_client(client)
secrets_client = SecretsClient.from_client(client)
try:
    secrets_client.create_secret(
        workspace=WORKSPACE,
        body=PlatformSecretCreateRequest(
            name=OPENAI_KEY_SECRET,
            value=SecretStr(openai_api_key),
            description="OpenAI API key used by Harbor Codex trials.",
        ),
    )
except ConflictError:
    secrets_client.update_secret(
        workspace=WORKSPACE,
        name=OPENAI_KEY_SECRET,
        body=PlatformSecretUpdateRequest(
            value=SecretStr(openai_api_key),
            description="OpenAI API key used by Harbor Codex trials.",
        ),
    )
print(f"Platform: {BASE_URL} | Workspace: {WORKSPACE}")
print(f"Dataset: {DATASET}")
print(f"Codex: {MODEL_NAME} via OpenAI API")

## 1. Upload and register the dataset

Each task is stored as a self-contained `tar.gz` archive. The cell registers Tasks and a Taskset and returns immutable revision pins. Keep the taskset pin to evaluate the same definitions again.

This cell is rerunnable: if a Task or Taskset name already exists, registration retrieves and reuses that entity.


In [ ]:
receipt = upload_harbor_dataset(
    DATASET,
    client=client,
    workspace=WORKSPACE,
    taskset_name=TASKSET_NAME,
    register=True,
    replace=True,
)
assert receipt.taskset_ref is not None
TASKSET_REF = receipt.taskset_ref.root
print(f"Taskset pin: {TASKSET_REF}")
for member in receipt.members:
    assert member.task_ref is not None
    print(f"\n{member.native_name}")
    print(f"  Task:    {member.task_ref.root}")
    print(f"  Archive: {member.definition.source.fileset_ref}")

### What is stored in Files?

```text
FileSet: <workspace>/harbor-tasksets
└── harbor_dataset/
    ├── greet-universe/<archive-sha256>/task_archive
    ├── sum-three/<archive-sha256>/task_archive
    └── debug-agent-runtime-error/<archive-sha256>/task_archive
```

Each archive contains its named native task directory, including `task.toml`, instructions, environment, and tests. The archive hash identifies the compressed bytes; the Task and Taskset revision hashes identify database entity content. The taskset holds pinned Task references, not copies of their archives.


## 2. Submit an evaluation

The `tasks` field accepts the taskset pin returned above. Harbor installs the pinned Codex CLI inside each Docker task environment. `env_secrets` resolves the OpenAI credential into the evaluator job and forwards it by name without persisting its value in the job specification or Harbor configuration. With no `OPENAI_BASE_URL`, Codex uses the OpenAI API directly.

Run this cell once per desired evaluation. Polling and reading results below do not submit another job.


In [ ]:
job = evaluator_client.submit_agent_eval_job(
    workspace=WORKSPACE,
    body=SubmitAgentEvalJobRequest(
        spec={
            "tasks": TASKSET_REF,
            "target": {
                "kind": "harbor",
                "agent_name": "codex",
                "agent_model_name": MODEL_NAME,
                "agent_kwargs": {"version": CODEX_VERSION},
                "env_secrets": {
                    "OPENAI_API_KEY": f"{WORKSPACE}/{OPENAI_KEY_SECRET}",
                },
                "n_attempts": 1,
                "n_concurrent_trials": 1,
            },
        },
    ),
).data()
JOB_NAME = job.name
print(f"Submitted: {JOB_NAME}")

## 3. Wait for completion

Execution typically takes around 1–2 minutes; the first run may take longer while Docker images build. This cell prints status changes and waits up to 15 minutes. A notebook timeout does not cancel the job: rerun this cell to continue waiting for the same job.


In [ ]:
ansi_escape = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")


def recent_job_logs():
    try:
        return (
            evaluator_client.list_agent_eval_job_logs(workspace=WORKSPACE, name=JOB_NAME, query_params={"tail": 100})
            .page()
            .items
        )
    except Exception:
        return []  # Status polling must still work if logs are temporarily unavailable.


deadline = time.monotonic() + 900
previous_status = None
while True:
    status = evaluator_client.get_agent_eval_job_status(
        workspace=WORKSPACE,
        name=JOB_NAME,
    ).data()
    if status.status.value != previous_status:
        print(f"{JOB_NAME}: {status.status.value}")
        previous_status = status.status.value
    if status.status.value in {"completed", "error", "cancelled"}:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError(f"{JOB_NAME} is still running. Rerun this cell to keep waiting.")
    time.sleep(2)

if status.status.value != "completed":
    logs = recent_job_logs()
    failed_tasks = [task for step in status.steps for task in step.tasks]
    causes = []
    for task in failed_tasks:
        error_stack = task.error_stack
        if error_stack is None:
            continue
        matches = re.findall(
            r"(?:ModuleNotFoundError|ValueError|RuntimeError|FileNotFoundError):[^|]+",
            error_stack,
        )
        causes.append(matches[-1].strip() if matches else error_stack[-500:].strip())
    error_message = (status.error_details or {}).get("message", "unknown error")
    display({"status": status.status.value, "error": error_message, "causes": causes})
    if logs:
        print("Recent job logs:")
        for log in logs[-8:]:
            message = ansi_escape.sub("", log.message).replace("\n", " ")
            print(f"  [{log.job_step}/{log.job_task}] {message[:500]}")
    cause = causes[-1] if causes else error_message
    raise RuntimeError(f"Job {JOB_NAME} ended with {status.status.value}: {cause}")

## 4. Read the saved results

A completed job means evaluation finished; it does **not** mean every task passed. The results bundle contains trial outcomes and scores. Read its JSONL members directly without extracting files onto your machine.


In [ ]:
payload = evaluator_client.download_agent_eval_job_result(
    workspace=WORKSPACE,
    job=JOB_NAME,
    name="agent-eval-results",
).read()

with tarfile.open(fileobj=io.BytesIO(payload), mode="r:*") as artifact:

    def read_records(filename):
        member = next(m for m in artifact.getmembers() if Path(m.name).name == filename)
        stream = artifact.extractfile(member)
        if stream is None:
            raise ValueError(f"Missing result file: {filename}")
        with stream:
            return [json.loads(line) for line in stream if line.strip()]

    trials = read_records("trials.jsonl")
    scores = read_records("scores.jsonl")

summary = ["| Task | Trial status | Score records |", "| --- | --- | --- |"]
for trial in trials:
    task_scores = [score for score in scores if score.get("task_id") == trial["task_id"]]
    summary.append(f"| {trial['task_id']} | {trial.get('status')} | {len(task_scores)} |")
display(Markdown("\n".join(summary)))
print(f"{len(trials)} trials, {len(scores)} scores | Job: {JOB_NAME}")
print("\nArtifact paths:")
for trial in trials:
    trial_dir = trial.get("metadata", {}).get("harbor_trial_dir")
    print(f"  {trial['task_id']}: {trial_dir}")

### Inspect scores and errors

Expand the structured records below to see reward values and any execution errors. The debug fixture deliberately emits reward `0.0` and stops before its sentinel second step; the two file-writing tasks should each emit reward `1.0`.


In [ ]:
display({"scores": scores})
trial_errors = [
    {"task_id": trial["task_id"], "status": trial.get("status"), "error": trial.get("error")}
    for trial in trials
    if trial.get("error") or trial.get("status") != "completed"
]
display({"trial_errors": trial_errors})

assert len(trial_errors) == 1
assert trial_errors[0]["task_id"] == "hello/debug-agent-runtime-error"
assert trial_errors[0]["status"] == "partial"
assert trial_errors[0]["error"]["type"] == "AgentTimeoutError"

## Keep or clean up

Keep `TASKSET_REF` and `JOB_NAME` for inspection and repeat evaluations. Resources are retained by default. Tasksets do not own their members' archives, so do not delete shared Filesets to clean up this example.

The optional cell below deletes only this Taskset head; Tasks, archives, revisions, and job results remain.


In [ ]:
DELETE_TASKSET = False
if DELETE_TASKSET:
    from nemo_evaluator.sdk.taskset_resources import EvaluatorTasksetsResource

    EvaluatorTasksetsResource(evaluator_client).delete(TASKSET_NAME, workspace=WORKSPACE)
    print(f"Deleted Taskset head: {WORKSPACE}/{TASKSET_NAME}")